# 🧠 Fase 2: Filtragem Baseada em Conteúdo (Content-Based Filtering)\nNeste notebook, vamos desmistificar o \"motor\" do Sistema de Recomendação. \nVamos construir a matemática passo a passo: desde a transformação de texto em números (TF-IDF) até a montagem do perfil do usuário com base no que ele curtiu (+) e rejeitou (-).\n

In [ ]:
import pandas as pd\nimport numpy as np\nimport matplotlib.pyplot as plt\nimport seaborn as sns\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.metrics.pairwise import cosine_similarity\nimport warnings\nwarnings.filterwarnings('ignore')\n\n# Configuração visual\nsns.set_theme(style=\"whitegrid\")\n

## 1. Carregamento dos Dados\nVamos carregar nossa base de `postings.csv`. Para focar na qualidade do recomendador e não estourar a memória (já que o TF-IDF cria matrizes gigantescas), vamos filtrar apenas vagas ativas (vamos limitar a uma amostra limpa) e com informações essenciais preenchidas.\n

In [ ]:
base_path = \"archive\" if os.path.exists(\"archive\") else \".\"\n\n# Carrega apenas as colunas necessárias para economizar RAM\ncolunas = ['job_id', 'title', 'skills_desc', 'formatted_experience_level', 'remote_allowed', 'applies', 'views']\ndf = pd.read_csv(f\"{base_path}/postings.csv\", usecols=lambda c: c in colunas)\n\n# Limpeza e Tratamento\ndf = df.dropna(subset=['title'])\n\n# Preenche nulos nas skills\ndf['skills_desc'] = df['skills_desc'].fillna('')\ndf['formatted_experience_level'] = df['formatted_experience_level'].fillna('')\n\n# Calcula o CTR que definimos na Fase 1\ndf['applies'] = df['applies'].fillna(0)\ndf['views'] = df['views'].fillna(1) # evita divisão por zero\ndf['ctr'] = df['applies'] / df['views']\ndf['ctr'] = df['ctr'].clip(upper=1.0) # Limita candidaturas externas anômalas\n\n# Flag de remoto\ndf['is_remote'] = df['remote_allowed'].fillna(0).astype(int)\n\n# Pega uma amostra de 25 mil vagas para manter o processamento rápido neste laboratório\ndf = df.sample(n=min(25000, len(df)), random_state=42).reset_index(drop=True)\nprint(f\"Base de laboratório pronta com {len(df)} vagas.\")\n

## 2. A \"Sopa de Palavras\" (Metadados do Item)\nEm recomendação baseada em conteúdo, precisamos representar o item (a vaga) como um documento de texto. \nComo o **título** é o atributo mais forte, daremos peso duplo a ele na concatenação.\n$$ \\text{Documento}_{i} = \\text{Título} \\times 2 + \\text{Skills} + \\text{Nível} $$\n

In [ ]:
def build_item_string(row):\n    # Peso 2 para o título para forçar a semântica principal\n    title_str = (str(row['title']) + \" \") * 2 \n    skills_str = str(row['skills_desc'])\n    level_str = str(row['formatted_experience_level']).replace(\" \", \"\") # Junta \"Entry level\" para \"Entrylevel\"\n    \n    return f\"{title_str} {skills_str} {level_str}\".lower()\n\ndf['item_string'] = df.apply(build_item_string, axis=1)\n\n# Vejamos como ficou o \"documento\" da primeira vaga:\nprint(\"Exemplo de Representação Vetorial de uma Vaga:\")\nprint(\"-\" * 50)\nprint(df[['title', 'item_string']].iloc[0]['item_string'])\n

## 3. Vetorização (TF-IDF)\nA matemática entra aqui. O TF-IDF (*Term Frequency-Inverse Document Frequency*) vai punir palavras que aparecem em todas as vagas (como \"and\", \"the\", \"work\") e dar um score altíssimo para palavras raras e específicas (como \"PyTorch\", \"Kubernetes\", \"B2B Sales\").\n

In [ ]:
# Configuramos n-gramas de (1,2) para pegar combinações como \"data scientist\" ou \"machine learning\"\ntfidf = TfidfVectorizer(\n    analyzer='word',\n    ngram_range=(1, 2),\n    min_df=5,           # Ignora termos que aparecem em menos de 5 vagas\n    max_features=5000,  # Limita aos 5000 termos mais relevantes do mercado\n    stop_words='english'\n)\n\n# Transforma a coluna de texto na grande matriz matemática (Vagas x Termos)\ntfidf_matrix = tfidf.fit_transform(df['item_string'])\n\nprint(f\"Dimensão da Matriz TF-IDF: {tfidf_matrix.shape}\")\nprint(f\"(Temos {tfidf_matrix.shape[0]} vagas representadas num espaço de {tfidf_matrix.shape[1]} dimensões de palavras)\")\n

## 4. O Perfil do Usuário (Matemática Pura)\nUm usuário não é nada além do histórico dele. Se ele gostou das vagas A e B, o perfil dele é a soma dos vetores de A e B. Se ele rejeitou a vaga C, nós subtraímos o vetor de C.\n$$ \\vec{u} = \\alpha \\sum_{i \\in I^+} \\vec{v}_i - \\beta \\sum_{j \\in I^-} \\vec{v}_j $$\nOnde $\\alpha$ é o peso do \"Gostei\" (ex: 1.0) e $\\beta$ é o peso do \"Não Gostei\" (ex: 0.5).\n

In [ ]:
def build_user_profile(positive_indices, negative_indices, matrix, alpha=1.0, beta=0.5):\n    \"\"\"\n    Constrói o vetor numérico do perfil do usuário.\n    \"\"\"\n    user_vector = np.zeros((1, matrix.shape[1]))\n    \n    # Soma os vetores positivos\n    if positive_indices:\n        positive_vectors = matrix[positive_indices]\n        # Soma todos os vetores ao longo do eixo das vagas, multiplicado pelo peso alfa\n        user_vector += alpha * np.asarray(positive_vectors.sum(axis=0))\n        \n    # Subtrai os vetores negativos\n    if negative_indices:\n        negative_vectors = matrix[negative_indices]\n        user_vector -= beta * np.asarray(negative_vectors.sum(axis=0))\n        \n    # Normalizamos o vetor para evitar que usuários muito ativos tenham scores muito maiores\n    # que usuários novos (apenas direção importa, não magnitude)\n    if np.linalg.norm(user_vector) > 0:\n        user_vector = user_vector / np.linalg.norm(user_vector)\n        \n    return user_vector\n

## 5. Função de Recomendação (Similaridade do Cosseno + CTR)\nPara cada vaga, vamos calcular o Cosseno do Ângulo entre o Vetor do Usuário e o Vetor da Vaga. \nQuanto mais próximo de 1, mais alinhado. Depois, multiplicamos pelo Bônus de CTR, aplicando a **Prudência Epistemológica** (valorizamos vagas com alta conversão empiricamente provada).\n

In [ ]:
def recommend_jobs(user_profile, matrix, df_data, top_n=10, ctr_weight=0.2, remote_only=False):\n    # Calcula a similaridade do cosseno entre o usuário e TODAS as vagas (retorna matriz 1 x N)\n    cosine_sim = cosine_similarity(user_profile, matrix).flatten()\n    \n    # Cria um DataFrame de resultados\n    results = df_data[['job_id', 'title', 'formatted_experience_level', 'is_remote', 'ctr']].copy()\n    results['similarity'] = cosine_sim\n    \n    # Modulador de CTR (Conforme nossa hipótese do EDA)\n    # Vagas muito populares ganham um bônus no score de até ctr_weight\n    results['final_score'] = results['similarity'] * (1.0 + (ctr_weight * results['ctr']))\n    \n    # Filtro Rígido\n    if remote_only:\n        results = results[results['is_remote'] == 1]\n        \n    # Ordena pelo score final\n    recommended = results.sort_values(by='final_score', ascending=False)\n    \n    return recommended.head(top_n)\n

## 6. Testando as Recomendações (Personas)\nVamos simular as interações para provar que a matemática funciona.\n

In [ ]:
# Buscando índices reais na base de dados para usarmos como \"Gostei\" / \"Não Gostei\"\n# Persona A: Cientista de Dados que odeia vagas de marketing\nds_vagas = df[df['title'].str.contains('Data Scientist', case=False, na=False)].index.tolist()[:3]\nmkt_vagas = df[df['title'].str.contains('Marketing', case=False, na=False)].index.tolist()[:2]\n\nprint(\"Vagas curtidas pela Persona A:\")\ndisplay(df.loc[ds_vagas, ['title', 'formatted_experience_level']])\n\nprint(\"\\nVagas REJEITADAS pela Persona A:\")\ndisplay(df.loc[mkt_vagas, ['title', 'formatted_experience_level']])\n

In [ ]:
# Gerando o Perfil\npersona_a_vector = build_user_profile(\n    positive_indices=ds_vagas, \n    negative_indices=mkt_vagas, \n    matrix=tfidf_matrix,\n    alpha=1.0, \n    beta=1.0 # Penalidade severa\n)\n\n# Recomendação sem filtro de remoto\nrecs_a = recommend_jobs(persona_a_vector, tfidf_matrix, df, top_n=5)\nprint(\"🎯 TOP 5 RECOMENDAÇÕES PARA A PERSONA A (Data Scientist):\")\ndisplay(recs_a)\n

In [ ]:
# Persona B: RH buscando vagas de recrutamento remoto\nhr_vagas = df[df['title'].str.contains('Recruiter|Human Resources', case=False, na=False)].index.tolist()[:3]\ndev_vagas = df[df['title'].str.contains('Software|Developer', case=False, na=False)].index.tolist()[:2]\n\npersona_b_vector = build_user_profile(\n    positive_indices=hr_vagas, \n    negative_indices=dev_vagas, \n    matrix=tfidf_matrix\n)\n\n# Recomendação exigindo filtro remoto\nrecs_b = recommend_jobs(persona_b_vector, tfidf_matrix, df, top_n=5, remote_only=True)\nprint(\"🎯 TOP 5 RECOMENDAÇÕES PARA A PERSONA B (Recrutador / Apenas Remoto):\")\ndisplay(recs_b)\n